### FORMULACIÓN DEL PROBLEMA Y MODELADO

In [16]:
from collections import deque
# Clase abstracta
class Problem:
    def __init__(self, initial, goal):
        self.initial = initial 
        self.goal = goal 
    
    def actions(self, state):
        raise NotImplementedError
# Función de transición
    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        return self.goal == state
# Función de desempeño   
    def action_cost(self, state1, action, state2):
        return 1
    
    def h(self, state):
        return 0

In [17]:
class GraphProblem(Problem):
    def __init__(self, initial, goal, graph):
        super().__init__(initial, goal)
        self.graph = graph
    
    def actions(self, state):
        lista = []
        for key in self.graph[state].keys():
            lista.append(key)
        return lista

    def result(self, state, action):
        return action

    def action_cost(self, state1, action, state2):
        return self.graph[state1][state2]

In [18]:
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self):
        lista_path = []
        node = self
        while node:
            lista_path.append(node.state)
            node = node.parent
        return lista_path[::-1]

    def expand(self, problem):
        lista = []
        for action in problem.actions(self.state):
            lista.append(self.child_node(problem, action))
        return lista

    def child_node(self, problem, action):
        next_state = problem.result(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)
        return Node(next_state, self, action, self.path_cost + step_cost)

### ALGORITMOS DE BÚSQUEDA

In [19]:
def depth_first_graph_search(problem):
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node
    frontier = [start_node]
    explored = set()

    while frontier:
        node = frontier.pop()
        explored.add(node.state)
        if problem.is_goal(node.state):
            return node
        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)
    return None

In [20]:
def breadth_first_graph_search(problem):
    start_node = Node(problem.initial)
    if problem.is_goal(start_node.state):
        return start_node
    frontier = deque([start_node])
    explored = set()

    while frontier:
        node = frontier.popleft()
        explored.add(node.state)
        if problem.is_goal(node.state):
            return node
        for child in node.expand(problem):
            if child.state not in explored:
                frontier.append(child)
    return None

### MAPA DEL METRO

In [21]:
metro_cdmx = {
    # LÍNEA 1 (Rosa)
    'Observatorio': {'Tacubaya': 1},
    'Tacubaya': {'Observatorio': 1, 'Juanacatlan': 1, 'Tacuba': 1, 'Constituyentes': 1, 'Patriotismo': 1}, # Transbordo L1, L7, L9
    'Juanacatlan': {'Tacubaya': 1, 'Chapultepec': 1},
    'Chapultepec': {'Juanacatlan': 1, 'Sevilla': 1},
    'Sevilla': {'Chapultepec': 1, 'Insurgentes': 1},
    'Insurgentes': {'Sevilla': 1, 'Cuauhtemoc': 1},
    'Cuauhtemoc': {'Insurgentes': 1, 'Balderas': 1},
    'Balderas': {'Cuauhtemoc': 1, 'Salto del Agua': 1, 'Juarez': 1, 'Niños Heroes': 1}, # Transbordo L1, L3
    'Salto del Agua': {'Balderas': 1, 'Isabel la Catolica': 1},
    'Isabel la Catolica': {'Salto del Agua': 1, 'Pino Suarez': 1},
    'Pino Suarez': {'Isabel la Catolica': 1, 'Merced': 1, 'Zocalo': 1, 'San Antonio Abad': 1}, # Transbordo L1, L2
    'Merced': {'Pino Suarez': 1, 'Candelaria': 1},
    'Candelaria': {'Merced': 1, 'San Lazaro': 1},
    'San Lazaro': {'Candelaria': 1, 'Moctezuma': 1},
    'Moctezuma': {'San Lazaro': 1, 'Balbuena': 1},
    'Balbuena': {'Moctezuma': 1, 'Boulevard Puerto Aereo': 1},
    'Boulevard Puerto Aereo': {'Balbuena': 1, 'Gomez Farias': 1},
    'Gomez Farias': {'Boulevard Puerto Aereo': 1, 'Zaragoza': 1},
    'Zaragoza': {'Gomez Farias': 1, 'Pantitlan': 1},
    'Pantitlan': {'Zaragoza': 1, 'Hangares': 1, 'Puebla': 1}, # Transbordo L1, L5, L9

    # LÍNEA 2 (Azul)
    'Cuatro Caminos': {'Panteones': 1},
    'Panteones': {'Cuatro Caminos': 1, 'Tacuba': 1},
    'Tacuba': {'Panteones': 1, 'Cuitlahuac': 1, 'Refineria': 1, 'San Joaquin': 1}, # Transbordo L2, L7
    'Cuitlahuac': {'Tacuba': 1, 'Popotla': 1},
    'Popotla': {'Cuitlahuac': 1, 'Colegio Militar': 1},
    'Colegio Militar': {'Popotla': 1, 'Normal': 1},
    'Normal': {'Colegio Militar': 1, 'San Cosme': 1},
    'San Cosme': {'Normal': 1, 'Revolucion': 1},
    'Revolucion': {'San Cosme': 1, 'Hidalgo': 1},
    'Hidalgo': {'Revolucion': 1, 'Bellas Artes': 1, 'Guerrero': 1, 'Juarez': 1}, # Transbordo L2, L3
    'Bellas Artes': {'Hidalgo': 1, 'Allende': 1},
    'Allende': {'Bellas Artes': 1, 'Zocalo': 1},
    'Zocalo': {'Allende': 1, 'Pino Suarez': 1},
    'San Antonio Abad': {'Pino Suarez': 1, 'Chabacano': 1},
    'Chabacano': {'San Antonio Abad': 1, 'Viaducto': 1, 'Lázaro Cárdenas': 1, 'Jamaica': 1}, # Transbordo L2, L9
    'Viaducto': {'Chabacano': 1, 'Xola': 1},
    'Xola': {'Viaducto': 1, 'Villa de Cortes': 1},
    'Villa de Cortes': {'Xola': 1, 'Nativitas': 1},
    'Nativitas': {'Villa de Cortes': 1, 'Portales': 1},
    'Portales': {'Nativitas': 1, 'Ermita': 1},
    'Ermita': {'Portales': 1, 'General Anaya': 1},
    'General Anaya': {'Ermita': 1, 'Taxqueña': 1},
    'Taxqueña': {'General Anaya': 1},

    # LÍNEA 3 (Verde Musgo)
    'Indios Verdes': {'Deportivo 18 de Marzo': 1},
    'Deportivo 18 de Marzo': {'Indios Verdes': 1, 'Potrero': 1},
    'Potrero': {'Deportivo 18 de Marzo': 1, 'La Raza': 1},
    'La Raza': {'Potrero': 1, 'Tlatelolco': 1, 'Autobuses del Norte': 1, 'Misterios': 1}, # Transbordo L3, L5
    'Tlatelolco': {'La Raza': 1, 'Guerrero': 1},
    'Guerrero': {'Tlatelolco': 1, 'Hidalgo': 1},
    'Juarez': {'Hidalgo': 1, 'Balderas': 1},
    'Niños Heroes': {'Balderas': 1, 'Hospital General': 1},
    'Hospital General': {'Niños Heroes': 1, 'Centro Medico': 1},
    'Centro Medico': {'Hospital General': 1, 'Etiopia': 1, 'Chilpancingo': 1, 'Lázaro Cárdenas': 1}, # Transbordo L3, L9
    'Etiopia': {'Centro Medico': 1, 'Eugenia': 1},
    'Eugenia': {'Etiopia': 1, 'Division del Norte': 1},
    'Division del Norte': {'Eugenia': 1, 'Zapata': 1},
    'Zapata': {'Division del Norte': 1, 'Coyoacan': 1},
    'Coyoacan': {'Zapata': 1, 'Viveros': 1},
    'Viveros': {'Coyoacan': 1, 'Miguel Angel de Quevedo': 1},
    'Miguel Angel de Quevedo': {'Viveros': 1, 'Copilco': 1},
    'Copilco': {'Miguel Angel de Quevedo': 1, 'Universidad': 1},
    'Universidad': {'Copilco': 1},

    # LÍNEA 5 (Amarilla)
    'Politecnico': {'Instituto del Petroleo': 1},
    'Instituto del Petroleo': {'Politecnico': 1, 'Autobuses del Norte': 1},
    'Autobuses del Norte': {'Instituto del Petroleo': 1, 'La Raza': 1},
    'Misterios': {'La Raza': 1, 'Valle Gomez': 1},
    'Valle Gomez': {'Misterios': 1, 'Consulado': 1},
    'Consulado': {'Valle Gomez': 1, 'Eduardo Molina': 1},
    'Eduardo Molina': {'Consulado': 1, 'Aragon': 1},
    'Aragon': {'Eduardo Molina': 1, 'Oceania': 1},
    'Oceania': {'Aragon': 1, 'Terminal Aerea': 1},
    'Terminal Aerea': {'Oceania': 1, 'Hangares': 1},
    'Hangares': {'Terminal Aerea': 1, 'Pantitlan': 1},

    # LÍNEA 7 (Naranja)
    'El Rosario': {'Aquiles Serdan': 1},
    'Aquiles Serdan': {'El Rosario': 1, 'Camarones': 1},
    'Camarones': {'Aquiles Serdan': 1, 'Refineria': 1},
    'Refineria': {'Camarones': 1, 'Tacuba': 1},
    'San Joaquin': {'Tacuba': 1, 'Polanco': 1},
    'Polanco': {'San Joaquin': 1, 'Auditorio': 1},
    'Auditorio': {'Polanco': 1, 'Constituyentes': 1},
    'Constituyentes': {'Auditorio': 1, 'Tacubaya': 1},
    'San Pedro de los Pinos': {'Tacubaya': 1, 'San Antonio': 1},
    'San Antonio': {'San Pedro de los Pinos': 1, 'Mixcoac': 1},
    'Mixcoac': {'San Antonio': 1, 'Barranca del Muerto': 1},
    'Barranca del Muerto': {'Mixcoac': 1},

    # LÍNEA 9 (Café)
    'Patriotismo': {'Tacubaya': 1, 'Chilpancingo': 1},
    'Chilpancingo': {'Patriotismo': 1, 'Centro Medico': 1},
    'Lázaro Cárdenas': {'Centro Medico': 1, 'Chabacano': 1},
    'Jamaica': {'Chabacano': 1, 'Mixiuhca': 1},
    'Mixiuhca': {'Jamaica': 1, 'Velodromo': 1},
    'Velodromo': {'Mixiuhca': 1, 'Ciudad Deportiva': 1},
    'Ciudad Deportiva': {'Velodromo': 1, 'Puebla': 1},
    'Puebla': {'Ciudad Deportiva': 1, 'Pantitlan': 1}
}

### RESULTADO

In [22]:
def imprimir_resultado(algoritmo_nombre, nodo_solucion):
    if nodo_solucion:
        camino = " -> ".join(nodo_solucion.path())
        costo = nodo_solucion.path_cost
        print(f"  {algoritmo_nombre} ({costo} estaciones):")
        print(f"  {camino}\n")
    else:
        print(f"  {algoritmo_nombre}: Mapa incompleto, no se encontró ruta.\n")


# --- RUTA 1 ---
print(" RUTA 1: Cuatro Caminos -> Pantitlán")
prob1 = GraphProblem('Cuatro Caminos', 'Pantitlan', metro_cdmx)

imprimir_resultado("DFS", depth_first_graph_search(prob1))
imprimir_resultado("BFS", breadth_first_graph_search(prob1))
print("\n")


# --- RUTA 2 ---
print(" RUTA 2: Politécnico -> Taxqueña")
prob2 = GraphProblem('Politecnico', 'Taxqueña', metro_cdmx)

imprimir_resultado("DFS", depth_first_graph_search(prob2))
imprimir_resultado("BFS", breadth_first_graph_search(prob2))
print("\n")


# --- RUTA 3 ---
print(" RUTA 3: Zapata -> Oceanía")
prob3 = GraphProblem('Zapata', 'Oceania', metro_cdmx)

imprimir_resultado("DFS", depth_first_graph_search(prob3))
imprimir_resultado("BFS", breadth_first_graph_search(prob3))
print("\n")

 RUTA 1: Cuatro Caminos -> Pantitlán
  DFS (18 estaciones):
  Cuatro Caminos -> Panteones -> Tacuba -> San Joaquin -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Medico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velodromo -> Ciudad Deportiva -> Puebla -> Pantitlan

  BFS (18 estaciones):
  Cuatro Caminos -> Panteones -> Tacuba -> San Joaquin -> Polanco -> Auditorio -> Constituyentes -> Tacubaya -> Patriotismo -> Chilpancingo -> Centro Medico -> Lázaro Cárdenas -> Chabacano -> Jamaica -> Mixiuhca -> Velodromo -> Ciudad Deportiva -> Puebla -> Pantitlan



 RUTA 2: Politécnico -> Taxqueña
  DFS (26 estaciones):
  Politecnico -> Instituto del Petroleo -> Autobuses del Norte -> La Raza -> Misterios -> Valle Gomez -> Consulado -> Eduardo Molina -> Aragon -> Oceania -> Terminal Aerea -> Hangares -> Pantitlan -> Puebla -> Ciudad Deportiva -> Velodromo -> Mixiuhca -> Jamaica -> Chabacano -> Viaducto -> Xola -> Villa de Cortes -> Nat